# Week 4 Tutorial: LLM Benchmarking vs Week 3 ML

This tutorial is intentionally simple.

You will:
- Load Week 3 test data
- Query three LLM endpoints
- Parse outputs into structured predictions
- Compare ML vs each LLM
- Save raw and clean result CSV files

## Step 1: Imports

What this does: imports the tools needed for file paths, table operations, parsing model output, and calling the model endpoint.

Why it matters: each import supports one part of the benchmark pipeline, so your notebook stays organized and predictable.

Principle: separate concerns. Use specific libraries for specific jobs (data handling, parsing, model calls) instead of mixing everything together.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

## Step 2: Load data and adding features

In [2]:
# ──────────────────────────────────────────────
# LOAD DATA
# ──────────────────────────────────────────────

count_by_class = pd.read_csv("region_yr_extreme_count.csv")
count_by_class = pd.pivot_table(count_by_class, values = 'n_extreme', index = ['NERC_ID', 'start_yr'], columns = 'extreme_type').astype('Int64').reset_index().fillna(0)

annual = count_by_class.rename(columns = {'NERC_ID': 'region', 'start_yr': 'year'})
annual.to_csv("annual_feature_matrix.csv", index=False)

#print(f"Annual feature matrix shape: {annual.shape}")
annual.head()

extreme_type,region,year,cold_dur_temp11,cold_dur_temp12,cold_dur_temp13,cold_dur_temp14,cold_dur_temp21,cold_dur_temp22,cold_dur_temp23,cold_dur_temp24,...,heat_dur_temp23,heat_dur_temp24,heat_dur_temp31,heat_dur_temp32,heat_dur_temp33,heat_dur_temp34,heat_dur_temp41,heat_dur_temp42,heat_dur_temp43,heat_dur_temp44
0,NERC1,1980,0,0,0,0,0,0,1,2,...,1,2,0,0,0,0,0,0,0,1
1,NERC1,1981,0,0,0,1,0,0,0,1,...,0,1,0,0,2,1,0,0,0,0
2,NERC1,1982,0,0,0,1,2,1,1,0,...,1,0,1,2,0,0,1,0,0,0
3,NERC1,1983,0,2,0,1,0,2,1,0,...,0,0,0,1,0,0,0,1,0,0
4,NERC1,1984,0,0,2,1,0,1,0,0,...,0,0,3,0,0,0,0,0,1,0


## Step 3: Paths and settings

What this does: sets all configurable values in one place: input files, column names, model settings, and output file locations.

Why it matters: if your team dataset or naming changes, you only edit one cell.

Principle: parameterization improves reproducibility and reduces accidental bugs from hardcoded values scattered across cells.

In [3]:
ROOT = Path('..').resolve()

WEEK3_DATA_CSV = 'annual_feature_matrix.csv'
WEEK3_RESULT_CSV = 'predict_test_from_m5.4tv.csv'

MODEL_ENDPOINTS = [
#    {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
#    {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

#PROMPT_DIR = ROOT / 'Prompts'
PROMPT_DIR = Path('../Prompts')
PROMPT_DIR.mkdir(parents=True, exist_ok=True)
#OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR = Path('../Results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_RESULTS_PATH = OUTPUT_DIR / f'llm_results_raw_{ITERATION_NUMBER}.csv'
CLEAN_RESULTS_PATH = OUTPUT_DIR / f'llm_results_clean_{ITERATION_NUMBER}.csv'

## Step 4: Load Week 3 files

What this does: loads Week 3 test data and Week 3 ML predictions, validates expected columns, then merges them on sample_id.

Why it matters: Week 4 comparison is only fair if both ML and LLM use the same held-out test examples.

Principle: benchmark validity depends on data integrity. Always validate schema and join keys before scoring.

In [4]:
data_df = pd.read_csv(WEEK3_DATA_CSV)

result_df = pd.read_csv(WEEK3_RESULT_CSV)
result_df = pd.pivot_table(result_df, values = 'pred_m5.4tv', index = ['NERC_ID', 'start_yr'], columns = 'extreme_type').reset_index()

data_df = data_df.merge(result_df, left_on = ["region", "year"], right_on = ["NERC_ID", "start_yr"], suffixes = ('', '_reg'), how = "outer").fillna(0).drop(columns = ["NERC_ID", "start_yr"])
data_df = data_df.rename(columns = {'cold_event_count': 'f1', 'cold_mean_severity': 'f2', 'cold_mean_duration': 'f3', 'heat_event_count': 'f4', 'heat_mean_severity': 'f5', 'heat_mean_duration': 'f6'})
for c in data_df.columns:
    if c.startswith('cold_dur_temp'): data_df = data_df.rename(columns = {c: f'c{c[13:]}'})
    elif c.startswith('heat_dur_temp'): data_df = data_df.rename(columns = {c: f'h{c[13:]}'})
data_df['region'] = data_df['region'].str.replace('ERC', '')

# iterate over region
nerc_sel = list(data_df['region'].unique())
#nerc_sel = ['N1', 'N2', 'N10', 'N11', 'N15']
#nerc_sel = ['N3', 'N8', 'N15']

# columns to be ground truth and predicted by llm
group_pred = [c for c in data_df.columns if c.startswith('c') and not c.endswith('_reg')]
# columns to be predicted by regression
group_reg = [c for c in data_df.columns if c.endswith('_reg')]
# columns to be additionally considered
group_add = []
##group_add = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6']

data_df[group_pred] = data_df[group_pred].astype(float).round(2)

data_df = data_df[data_df['region'].isin(nerc_sel)]
train_df = data_df[data_df['year'] < 2019]
test_df = data_df[data_df['year'] >= 2019]

print('Training Data:')
display(train_df.head())
print('Test Data:')
display(test_df.head())

Training Data:


,region,year,c11,c12,c13,c14,c21,c22,c23,c24,...,c23_reg,c24_reg,c31_reg,c32_reg,c33_reg,c34_reg,c41_reg,c42_reg,c43_reg,c44_reg
0,N1,1980,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,N1,1981,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,N1,1982,0.0,0.0,0.0,1.0,2.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,N1,1983,0.0,2.0,0.0,1.0,0.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,N1,1984,0.0,0.0,2.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Test Data:


,region,year,c11,c12,c13,c14,c21,c22,c23,c24,...,c23_reg,c24_reg,c31_reg,c32_reg,c33_reg,c34_reg,c41_reg,c42_reg,c43_reg,c44_reg
39,N1,2019,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.448988,0.384841,0.496664,0.577464,0.432496,0.196018,1.179864,0.473658,0.153364,0.016177
40,N1,2020,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,...,0.457487,0.408246,0.497659,0.588589,0.441785,0.201915,1.182123,0.483480,0.157373,0.017293
41,N1,2021,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,...,0.466146,0.433075,0.498655,0.599929,0.451275,0.207989,1.184386,0.493506,0.161487,0.018485
42,N1,2022,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.474970,0.459414,0.499653,0.611486,0.460968,0.214246,1.186653,0.503740,0.165709,0.019760
43,N1,2023,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,...,0.483961,0.487355,0.500653,0.623267,0.470869,0.220692,1.188925,0.514185,0.170041,0.021123


## Step 5-1: Prompt + parser helpers

What this does: defines a strict prompt format and a parser that turns model text into structured fields (prediction, confidence, parse status).

Why it matters: LLM responses are free-form by default, but evaluation requires deterministic, machine-readable outputs.

Principle: constrain generation and validate parsing. Reliability comes from clear output contracts plus defensive checks.

## Step 5-2: Define prompt versions

What this does: creates alternative prompt templates (baseline and few-shot) and one parser for all outputs.

Why it matters: you can test formatting and accuracy differences while keeping evaluation logic constant.

Principle: change one variable at a time. Here, prompt text changes while parser and scoring stay fixed.

In [5]:
#label_values = sorted(df[LABEL_COL].astype(str).unique().tolist())
#label_set = set(label_values)

output_json = {c: ["predicted number", "confidence in 0-1"] for c in group_pred}

rng1 = np.random.default_rng(seed=1)
rng2 = np.random.default_rng(seed=2)
few_shot = [
#    'Example input 1: {"region": ' + "'" + f'{train_df["region"].iloc[0]}' + "'" + ', "year": ' + f'{train_df["year"].iloc[0]}' + '})\n',
#    'Example output 1: ' + json.dumps({c: [float(train_df[c].iloc[0]), 1.0] for c in group_pred}) + '\n',
    'Example input 1: {"region": ' + "'" + 'N13' + "'" + ', "year": ' + '2023' + '})\n',
    'Example output 1: ' + json.dumps({c: [float(rng1.integers(0, 10)), round(rng1.random(), 2)] for c in group_pred}) + '\n'
    'Example input 2: {"region": ' + "'" + 'N14' + "'" + ', "year": ' + '2024' + '})\n',
    'Example output 2: ' + json.dumps({c: [float(rng2.integers(0, 10)), round(rng2.random(), 2)] for c in group_pred}) + '\n'
]
few_shot = ''.join(few_shot)
print(few_shot)

def make_prompt(data, region, year, version = 'v1', few_shot = None):
    if few_shot is None: text_few_shot = ''
    else: text_few_shot = few_shot
        
    if version == 'v1':
        return (
            f'Prompt: Predict {group_pred} for the input.\n'
            'Output: Return ONLY JSON object with following this schema:\n'
            f'{output_json}\n'
            'NO EXTRA TEXT.\n'
            f'{text_few_shot}'
            f'Training Data: {data}\n'
            'Now Predict: {"region": ' + "'" + f'{region}' + "'" + ', "year": ' + f'{year}' + '})\n'
        )
    if version == 'v2':
        return (
            f'Prompt: Predict counts of classification {group_pred} by year for the input. Reduce mean absolute errors.\n'
            'Output: Return ONLY JSON object with following this schema:\n'
            f'{output_json}\n'
            'NO EXTRA TEXT.\n'
            f'{text_few_shot}'
            f'Training Data: {data}\n'
            'Now Predict: {"region": ' + "'" + f'{region}' + "'" + ', "year": ' + f'{year}' + '})\n'
        )

#def parse_response(raw_text, valid_labels):
def parse_response(raw_text):
#    output = {'llm_prediction': None, 'llm_confidence': np.nan, 'parse_ok': False, 'parse_error': None}
    output = {'llm_prediction': None, 'llm_confidence': None, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        print(candidate)
        output['parse_error'] = 'invalid_json'
        return output

#    pred = str(payload.get('prediction', '')).strip()
#    if pred not in valid_labels:
#        output['parse_error'] = 'invalid_label'
#        return output

#    conf = payload.get('confidence', np.nan)
#    try:
#        conf = float(conf)
#    except Exception:
#        conf = np.nan

#    if not np.isnan(conf) and not (0 <= conf <= 1):
#        conf = np.nan
    
    pred = {c: payload.get(c)[0] for c in group_pred}
    conf = {c: payload.get(c)[1] for c in group_pred}
    
    for c in group_pred:
        if c not in payload.keys():
            output['parse_error'] = 'invalid_group'
            return output
        try:
            float(payload.get(c)[0])
        except:
            output['parse_error'] = 'invalid_type'
            return output
        try:
            float(payload.get(c)[1])
        except:
            output['parse_error'] = 'invalid_type'
            return output
        if not (0 <= float(payload.get(c)[1]) <= 1):
            output['parse_error'] = 'invalid_range'
            return output

    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return json.dumps(output)


Example input 1: {"region": 'N13', "year": 2023})
Example output 1: {"c11": [4.0, 0.95], "c12": [5.0, 0.14], "c13": [8.0, 0.31], "c14": [9.0, 0.42], "c21": [2.0, 0.41], "c22": [8.0, 0.55], "c23": [0.0, 0.75], "c24": [0.0, 0.54], "c31": [8.0, 0.79], "c32": [3.0, 0.3], "c33": [1.0, 0.13], "c34": [4.0, 0.4], "c41": [9.0, 0.26], "c42": [2.0, 0.75], "c43": [0.0, 0.49], "c44": [2.0, 0.98]}
Example input 2: {"region": 'N14', "year": 2024})
Example output 2: {"c11": [8.0, 0.3], "c12": [2.0, 0.81], "c13": [4.0, 0.6], "c14": [0.0, 0.73], "c21": [9.0, 0.06], "c22": [1.0, 0.27], "c23": [2.0, 0.56], "c24": [6.0, 0.15], "c31": [7.0, 0.67], "c32": [4.0, 0.42], "c33": [2.0, 0.97], "c34": [6.0, 0.68], "c41": [3.0, 0.19], "c42": [3.0, 0.35], "c43": [5.0, 0.89], "c44": [5.0, 0.78]}



## Step 6: Single-row smoke test

What this does: tests one example on each model endpoint before running the full benchmark loop.

Why it matters: this catches endpoint/port issues early and confirms all models are reachable.

Principle: validate connectivity and output format for each model before large-scale evaluation.

In [6]:
SYSTEM_PROMPT = 'You are a regression model. Given the input features, output predicted numeric values.'

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

#rows_subset = slice(None, None) # select a training subset for smoke test
for endpoint_cfg in MODEL_ENDPOINTS:
    region = nerc_sel[0]
    year = 1981
    print(region, year)
    prompt = make_prompt(
        data = train_df.loc[train_df['region'] == region, ['region', 'year'] + group_add + group_pred].reset_index(drop=True).to_dict(orient='records'),
        region = region,
        year = year)
    raw = query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw)
    print(f"\nModel: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print('Raw response:')
    print(raw)
    print('Parsed:')
    print(parsed)

N1 1981

Model: gemma-3-12b-it (port 8002)
Raw response:
```json
{
  "c11": [0.0, 0.9],
  "c12": [0.0, 0.9],
  "c13": [0.0, 0.9],
  "c14": [1.0, 0.8],
  "c21": [0.0, 0.9],
  "c22": [0.0, 0.9],
  "c23": [0.0, 0.8],
  "c24": [1.0, 0.7],
  "c31": [0.0, 0.9],
  "c32": [2.0, 0.7],
  "c33": [0.0, 0.8],
  "c34": [0.0, 0.9],
  "c41": [0.0, 0.9],
  "c42": [0.0, 0.9],
  "c43": [0.0, 0.8],
  "c44": [0.0, 0.9]
}
```
Parsed:
{"llm_prediction": {"c11": 0.0, "c12": 0.0, "c13": 0.0, "c14": 1.0, "c21": 0.0, "c22": 0.0, "c23": 0.0, "c24": 1.0, "c31": 0.0, "c32": 2.0, "c33": 0.0, "c34": 0.0, "c41": 0.0, "c42": 0.0, "c43": 0.0, "c44": 0.0}, "llm_confidence": {"c11": 0.9, "c12": 0.9, "c13": 0.9, "c14": 0.8, "c21": 0.9, "c22": 0.9, "c23": 0.8, "c24": 0.7, "c31": 0.9, "c32": 0.7, "c33": 0.8, "c34": 0.9, "c41": 0.9, "c42": 0.9, "c43": 0.8, "c44": 0.9}, "parse_ok": true, "parse_error": null}


## Step 7: Evaluate prompt versions on a small subset

What this does: runs each prompt version on the same subset and computes parse success and accuracy.

Why it matters: small-scale testing is faster and helps pick a strong prompt before full benchmarking.

Principle: iterate quickly on a representative subset, then scale to the full test set once stable.

WARNING: Nested for loops + LLM queries = death by boredom and kernel crashes from time outs. In your actual testing keep subsets small and/or query one model at a time.

In [7]:
subset = test_df.sample(n=min(5, len(test_df)), random_state=7)#.reset_index(drop=True)
ver_and_fewshot = [('v1', None), ('v1', few_shot), ('v2', None), ('v2', few_shot)]

records = []
for endpoint_cfg in MODEL_ENDPOINTS:
    for version, few_shot in ver_and_fewshot:
        for _, row in subset.iterrows():
            prompt = make_prompt(
                data = train_df.loc[train_df['region'] == row['region'], ['region', 'year'] + group_add + group_pred].reset_index(drop=True).to_dict(orient='records'),
                region = row['region'],
                year = row['year'],
                version = version,
                few_shot = few_shot
            )
            
            raw = query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0)
            
            res = pd.DataFrame(json.loads(parse_response(raw))).reset_index().rename(columns = {'index': 'extreme_type'}).assign(region = row['region'], year = row['year']).set_index(['region', 'year'])
            res['llm_prediction'] = res['llm_prediction'].astype(float)
            res['llm_confidence'] = res['llm_confidence'].astype(float)

            # merge ground truth
            data_merge = data_df.loc[(data_df['region'] == row['region']) & (data_df['year'] == row['year']), group_pred].iloc[0].to_frame(name = 'ground_truth')
            res = res.merge(data_merge, left_on = 'extreme_type', right_index = True, how = 'outer')
            
            res['llm_diff'] = res['llm_prediction'] - res['ground_truth']

            records.append({
                'region': row['region'],
                'year': row['year'],
                'model_label': endpoint_cfg['label'],
                'model_name': endpoint_cfg['model_name'],
                'endpoint_port': endpoint_cfg['port'],
                'version': version,
                'few_shot': few_shot is None,
                'correct %': (res['llm_diff'] == 0).sum() / len(res) * 100,
                'MAE': res['llm_diff'].abs().mean(),
                'confidence': res['llm_confidence'].mean(),
                'parse_ok': bool(res['parse_ok'].iloc[0]),
                'parse_error': str(res['parse_error'].iloc[0]),
#                'raw_response': raw
            })

records_df = pd.DataFrame(records)
display(records_df)

summary = records_df.groupby(['model_label', 'model_name', 'endpoint_port', 'version', 'few_shot'], as_index=False).agg(
    correct_mean=('correct %', 'mean'),
    MAE_mean=('MAE', 'mean'),
    confidence_mean=('confidence', 'mean')
)
summary = summary.sort_values(['model_label', 'correct_mean', 'MAE_mean', 'confidence_mean'], ascending=[True, False, True, False])
display(summary)

,region,year,model_label,model_name,endpoint_port,version,few_shot,correct %,MAE,confidence,parse_ok,parse_error
0,N12,2021,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,75.00,0.2500,0.90000,True,None
1,N7,2023,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,62.50,0.6250,0.68125,True,None
2,N6,2024,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,81.25,0.1875,0.88125,True,None
3,N6,2020,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,75.00,0.4375,0.86875,True,None
4,N18,2020,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,75.00,0.3125,0.68125,True,None
5,N12,2021,gemma-3-12b-it,gemma-3-12b-it,8002,v1,False,75.00,0.3125,0.00000,True,None
6,N7,2023,gemma-3-12b-it,gemma-3-12b-it,8002,v1,False,62.50,0.6250,0.95000,True,None
7,N6,2024,gemma-3-12b-it,gemma-3-12b-it,8002,v1,False,75.00,0.2500,0.93750,True,None
8,N6,2020,gemma-3-12b-it,gemma-3-12b-it,8002,v1,False,62.50,0.5625,0.53750,True,None
9,N18,2020,gemma-3-12b-it,gemma-3-12b-it,8002,v1,False,75.00,0.3750,0.00000,True,None


,model_label,model_name,endpoint_port,version,few_shot,correct_mean,MAE_mean,confidence_mean
1,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,73.75,0.3625,0.80250
0,gemma-3-12b-it,gemma-3-12b-it,8002,v1,False,70.00,0.4250,0.48500
3,gemma-3-12b-it,gemma-3-12b-it,8002,v2,True,58.75,0.5250,0.69375
2,gemma-3-12b-it,gemma-3-12b-it,8002,v2,False,56.25,0.5250,0.59475


## Step 8: Save final prompt files

What this does: writes your selected prompt template and few-shot examples to the submission folder.

Why it matters: prompt artifacts make your workflow auditable and reproducible for judges.

Principle: version your prompts like code. Prompt files are part of the experiment definition.

In [8]:
best_model = summary.groupby('model_label', as_index=False).first()[['model_label', 'model_name', 'endpoint_port', 'version', 'few_shot']]
display(best_model)

# Save one canonical template + few-shot file for submission compatibility.
final_prompt_example = make_prompt(
    data = '[TRAIN DATA by REGION]',
    region = '[REGION]',
    year = '[YEAR]',
    version = best_model['version'].squeeze(),
    few_shot = few_shot if best_model['few_shot'].squeeze() else None
)
prompt_template_path = PROMPT_DIR / 'prompt_template.txt'
few_shot_path = PROMPT_DIR / 'few_shot_samples.txt'
prompt_template_path.write_text(final_prompt_example, encoding='utf-8')
few_shot_path.write_text(few_shot, encoding='utf-8')

,model_label,model_name,endpoint_port,version,few_shot
0,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True


774

## Step 9: Draft report text

What this does: creates starter language for approach and error-analysis sections.

Why it matters: structured drafting helps teams report results consistently and quickly.

Principle: automate repetitive reporting, then refine with concrete findings from your run.

In [9]:
display(summary)

approach_report = (
    'We evaluated one LLM endpoint after initially testing three endpoints; the two mini-LLM endpoints failed because of input/output token limitations. '
    'We compared multiple prompt versions on the same held-out subset from Week 3. '
    'To address the token limit issue, we attempted to shorten the prompt input by reducing the number of characters, but this did not address the issue with the two failed endpoints. '
    'We then selected the best-performing prompt from four candidates, based on two prompt versions and whether to include a few-shot example. '
    'Selection was based on parse reliability and accuracy. '
    'The outputs for event counts across the 16 classes were strictly constrained to JSON format and parsed before scoring. '
)

error_analysis = (
    'Common failures included malformed JSON, invalid value types or ranges, and inconsistent formatting across models. '
    'Prompt wording slightly affected compliance and output quality, but its overall impact is difficult to assess because of the limited regression scope, including cases where no event was predicted. '
    'Few-shot prompting often improved reliability and, in some cases, determined whether regression can be possible when testing the three LLM endpoints. '
)

print('Approach Report draft:\n')
print(approach_report)
print('\nError Analysis draft:\n')
print(error_analysis)

,model_label,model_name,endpoint_port,version,few_shot,correct_mean,MAE_mean,confidence_mean
1,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,73.75,0.3625,0.80250
0,gemma-3-12b-it,gemma-3-12b-it,8002,v1,False,70.00,0.4250,0.48500
3,gemma-3-12b-it,gemma-3-12b-it,8002,v2,True,58.75,0.5250,0.69375
2,gemma-3-12b-it,gemma-3-12b-it,8002,v2,False,56.25,0.5250,0.59475


Approach Report draft:

We evaluated one LLM endpoint after initially testing three endpoints; the two mini-LLM endpoints failed because of input/output token limitations. We compared multiple prompt versions on the same held-out subset from Week 3. To address the token limit issue, we attempted to shorten the prompt input by reducing the number of characters, but this did not address the issue with the two failed endpoints. We then selected the best-performing prompt from four candidates, based on two prompt versions and whether to include a few-shot example. Selection was based on parse reliability and accuracy. The outputs for event counts across the 16 classes were strictly constrained to JSON format and parsed before scoring. 

Error Analysis draft:

Common failures included malformed JSON, invalid value types or ranges, and inconsistent formatting across models. Prompt wording slightly affected compliance and output quality, but its overall impact is difficult to assess because o

## Step 10: Full benchmark run

What this does: loops over all configured model endpoints and all test rows, storing raw responses and parsed predictions.

Why it matters: this produces side-by-side model outputs from a single consistent pipeline.

Principle: hold the dataset and prompt constant while varying models for a fair model-to-model comparison.

In [10]:
clean_df = pd.DataFrame()
records = []
for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nRunning benchmark for {endpoint_cfg['model_name']} on port {endpoint_cfg['port']}...")
#    for i, row in subset.iterrows():
    for i, row in test_df.reset_index().iterrows():
        prompt = make_prompt(
            data = train_df.loc[train_df['region'] == row['region'], ['region', 'year'] + group_add + group_pred].reset_index(drop=True).to_dict(orient='records'),
            region = row['region'],
            year = row['year'],
            version = best_model['version'].squeeze(),
            few_shot = few_shot if best_model['few_shot'].squeeze() else None
        )

        raw = query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0)
        
        res = pd.DataFrame(json.loads(parse_response(raw))).reset_index().rename(columns = {'index': 'extreme_type'}).assign(region = row['region'], year = row['year']).set_index(['region', 'year'])
        res['llm_prediction'] = res['llm_prediction'].astype(float)
        res['llm_confidence'] = res['llm_confidence'].astype(float)
        res['parse_error'] = res['parse_error'].astype(str)
        
        # merge ground truth
        data_merge = data_df.loc[(data_df['region'] == row['region']) & (data_df['year'] == row['year']), group_pred].iloc[0].to_frame(name = 'ground_truth')
        res = res.merge(data_merge, left_on = 'extreme_type', right_index = True, how = 'outer')
        
        # merge prediction by regression
        data_merge = data_df.loc[(data_df['region'] == row['region']) & (data_df['year'] == row['year']), group_reg].iloc[0].to_frame(name = 'regression_prediction')
        data_merge.index = data_merge.index.str.replace('_reg', '')
        res = res.merge(data_merge, left_on = 'extreme_type', right_index = True, how = 'outer')
        
        # llm error
        res['llm_diff'] = res['llm_prediction'] - res['ground_truth']
        
        # regression error
        res['regression_diff'] = res['regression_prediction'] - res['ground_truth']
        
        clean_df = pd.concat([clean_df, res])

        records.append({
            'region': row['region'],
            'year': row['year'],
            'model_label': endpoint_cfg['label'],
            'model_name': endpoint_cfg['model_name'],
            'endpoint_port': endpoint_cfg['port'],
            'version': best_model['version'].squeeze(),
            'few_shot': True if best_model['few_shot'].squeeze() else None,
            'correct %': (res['llm_diff'] == 0).sum() / len(res) * 100,
            'MAE': res['llm_diff'].abs().mean(),
            'confidence': res['llm_confidence'].mean(),
            'parse_ok': bool(res['parse_ok'].iloc[0]),
            'parse_error': str(res['parse_error'].iloc[0]),
            'raw_response (first 100 char)': raw[:100] + '...'
        })

        if (i + 1) % 10 == 0:
#            print(f"  Completed {i + 1}/{len(subset)}")
            print(f"  Completed {i + 1}/{len(test_df)}")

records_df = pd.DataFrame(records)
records_df.to_csv(RAW_RESULTS_PATH, index=False)
print(f'Saved raw CSV: {RAW_RESULTS_PATH}')
display(records_df.head())

clean_df = clean_df.reset_index()


Running benchmark for gemma-3-12b-it on port 8002...
  Completed 10/96
  Completed 20/96
  Completed 30/96
  Completed 40/96
  Completed 50/96
  Completed 60/96
  Completed 70/96
  Completed 80/96
  Completed 90/96
Saved raw CSV: ../Results/llm_results_raw_1.csv


,region,year,model_label,model_name,endpoint_port,version,few_shot,correct %,MAE,confidence,parse_ok,parse_error,raw_response (first 100 char)
0,N1,2019,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,62.50,0.3750,0.443750,True,nan,"```json\n{""c11"": [0.0, 0.9], ""c12"": [0.0, 0.8]..."
1,N1,2020,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,50.00,0.6875,0.562500,True,nan,"```json\n{""c11"": [0.0, 0.9], ""c12"": [0.0, 0.8]..."
2,N1,2021,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,50.00,0.7500,0.537500,True,nan,"```json\n{""c11"": [0.0, 0.9], ""c12"": [0.0, 0.8]..."
3,N1,2022,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,62.50,0.4375,0.475000,True,nan,"```json\n{""c11"": [0.0, 0.9], ""c12"": [0.0, 0.8]..."
4,N1,2023,gemma-3-12b-it,gemma-3-12b-it,8002,v1,True,56.25,0.6250,0.710625,True,nan,"```json\n{""c11"": [0.0, 0.95], ""c12"": [0.0, 0.9..."


In [11]:
print(test_df)

    region  year  c11  c12  c13  c14  c21  c22  c23  c24  ...   c23_reg  \
39      N1  2019  0.0  0.0  0.0  0.0  0.0  0.0  1.0  1.0  ...  0.448988   
40      N1  2020  0.0  0.0  0.0  0.0  0.0  1.0  2.0  3.0  ...  0.457487   
41      N1  2021  0.0  0.0  0.0  0.0  0.0  1.0  2.0  3.0  ...  0.466146   
42      N1  2022  0.0  0.0  0.0  0.0  0.0  0.0  1.0  1.0  ...  0.474970   
43      N1  2023  0.0  0.0  0.0  0.0  0.0  0.0  2.0  2.0  ...  0.483961   
..     ...   ...  ...  ...  ...  ...  ...  ...  ...  ...  ...       ...   
715     N9  2020  0.0  0.0  0.0  0.0  0.0  1.0  4.0  0.0  ...  0.540741   
716     N9  2021  0.0  0.0  0.0  0.0  0.0  0.0  0.0  2.0  ...  0.550977   
717     N9  2022  0.0  0.0  0.0  0.0  0.0  1.0  4.0  1.0  ...  0.561407   
718     N9  2023  0.0  0.0  0.0  0.0  0.0  1.0  2.0  5.0  ...  0.572034   
719     N9  2024  0.0  0.0  0.0  0.0  0.0  1.0  2.0  2.0  ...  0.582862   

      c24_reg   c31_reg   c32_reg   c33_reg   c34_reg   c41_reg   c42_reg  \
39   0.384841  0.49666

## Step 11: Evaluate and compare

What this does: computes ML accuracy, LLM accuracy, and invalid output rate from the parsed results.

Why it matters: this gives a direct Week 3 vs Week 4 comparison on the same data split.

Principle: evaluate both quality and reliability. Accuracy without format reliability can still fail in production workflows.

In [12]:
clean_cols = [
    'region', 'year', 'extreme_type', 'ground_truth', 'llm_prediction', 'regression_prediction', 'llm_diff', 'regression_diff', 'llm_confidence', 'parse_ok', 'parse_error'
]
clean_df = clean_df[clean_cols]
clean_df.to_csv(CLEAN_RESULTS_PATH, index=False)
print(f'Saved clean CSV: {CLEAN_RESULTS_PATH}')
display(clean_df.head())

# calculate MAE over year by region and extreme_type
MAE_region_type = clean_df.reset_index().groupby(['region', 'extreme_type'])[['llm_diff', 'regression_diff']].apply(lambda x: x.abs().mean()).rename(columns = {'llm_diff': 'llm_MAE', 'regression_diff': 'regression_MAE'})

# aggreegate (sum) over extreme_type and calculate MAE over year by region
MAE_region = clean_df.reset_index().groupby(['region', 'year'])[['llm_diff', 'regression_diff']].sum()
MAE_region = MAE_region.reset_index().groupby('region')[['llm_diff', 'regression_diff']].apply(lambda x: x.abs().mean()).rename(columns = {'llm_diff': 'llm_MAE', 'regression_diff': 'regression_MAE'})

display(MAE_region_type.head(20))
display(MAE_region.head(20))

Saved clean CSV: ../Results/llm_results_clean_1.csv


,region,year,extreme_type,ground_truth,llm_prediction,regression_prediction,llm_diff,regression_diff,llm_confidence,parse_ok,parse_error
0,N1,2019,c11,0.0,0.0,0.054550,0.0,0.054550,0.9,True,NaN
1,N1,2019,c12,0.0,0.0,0.318951,0.0,0.318951,0.8,True,NaN
2,N1,2019,c13,0.0,0.0,0.730280,0.0,0.730280,0.7,True,NaN
3,N1,2019,c14,0.0,1.0,1.061722,1.0,1.061722,0.6,True,NaN
4,N1,2019,c21,0.0,0.0,0.201429,0.0,0.201429,0.7,True,NaN


llm_MAE  regression_MAE
region extreme_type                          
N1     c11           0.000000        0.047746
       c12           0.000000        0.290752
       c13           0.500000        0.667570
       c14           1.000000        0.974630
       c21           0.000000        0.183827
       c22           0.666667        0.479787
       c23           0.500000        1.029221
       c24           1.666667        1.385012
       c31           0.333333        0.668062
       c32           0.500000        0.546871
       c33           1.333333        1.033894
       c34           0.833333        0.832414
       c41           0.666667        0.789974
       c42           0.833333        0.668375
       c43           0.000000        0.163743
       c44           0.000000        0.019236
N10    c11           0.000000        0.057078
       c12           0.000000        0.347580
       c13           0.000000        0.798048
       c14           0.000000        1.165123

,llm_MAE,regression_MAE
region,,
N1,4.833333,2.397490
N10,6.666667,1.877288
N11,6.000000,2.838670
N12,3.166667,2.973450
N15,4.166667,2.827884
N17,2.833333,3.441790
N18,4.833333,2.934488
N2,6.166667,1.170530
N20,3.500000,3.010190


Done